# 10. Unified XGBoost Sensitivity Check

**Purpose:** The final report (`06`) uses `RandomForestClassifier` for the baseline and team-history models but `XGBClassifier` for the player-enriched model. This confound means we cannot cleanly attribute the macro-F1 difference to the player features alone. This notebook re-runs all three models with the **same XGBoost algorithm and hyperparameters**, and checks whether the core report conclusions survive.

**Design rule:** The only variables that differ across the three models are (a) the training window and (b) the feature set. Everything else—algorithm, hyperparameters, imputation, holdout split—is held constant.

**Decision criterion:** If the ordering of models (player_enriched ≥ team_history ≥ baseline on macro-F1) and the knockout-stage directional advantage hold under unified XGB, the original report conclusions are attributable to data rather than algorithm choice.

In [ ]:
import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

NOTEBOOK_DIR = Path.cwd()
DATA         = NOTEBOOK_DIR / "data"
DB_PATH      = NOTEBOOK_DIR / "../../../../data/raw/worldcup/data-sqlite/worldcup.db"
LOCK_DIR     = DATA / "final_report_lock"       # 06 locked outputs (read-only here)
OUT_DIR      = DATA / "unified_xgb_sensitivity"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED          = 42
HOLDOUT_YEARS = [2018, 2022]
BOOTSTRAP_B   = 5000
CLASS_LABELS  = ["fav_loses", "draw", "fav_wins"]
MODEL_KEYS    = ["baseline", "team_history", "player_enriched"]

print("NOTEBOOK_DIR :", NOTEBOOK_DIR)
print("LOCK_DIR     :", LOCK_DIR)
print("OUT_DIR      :", OUT_DIR)

## A. Rebuild the match modeling table (identical to 06)

In [ ]:
feat_team = pd.read_parquet(DATA / "features_team.parquet")
feat_ps   = pd.read_parquet(DATA / "features_player_squad.parquet")

con = sqlite3.connect(DB_PATH)
def _sql(q): return pd.read_sql_query(q, con)
matches       = _sql("SELECT * FROM matches")
tournaments   = _sql("SELECT tournament_id, tournament_name, year, host_country FROM tournaments")
teams_db      = _sql("SELECT team_id, confederation_id, team_name FROM teams")
con.close()

matches["match_dt"] = pd.to_datetime(matches["match_date"], unit="D", origin="unix", errors="coerce")
men_ids = tournaments.loc[
    ~tournaments["tournament_name"].str.contains("Women", case=False, na=False), "tournament_id"
].unique()
matches = matches[matches["tournament_id"].isin(men_ids)].copy()
matches = matches.merge(tournaments[["tournament_id", "year", "host_country"]], on="tournament_id", how="left")

def _norm(s):
    return s.astype(str).str.strip().str.lower().str.replace(" ", "", regex=False)

# Rest-day features within each tournament
sched = matches[["match_id", "tournament_id", "home_team_id", "away_team_id", "match_dt", "year"]].copy()
long = pd.concat(
    [sched.rename(columns={"home_team_id": "team_id", "away_team_id": "opp_id"}),
     sched.rename(columns={"away_team_id": "team_id", "home_team_id": "opp_id"})],
    ignore_index=True,
).sort_values(["tournament_id", "team_id", "match_dt", "match_id"])
long["prev_dt"]    = long.groupby(["tournament_id", "team_id"])["match_dt"].shift(1)
long["rest_days"]  = (long["match_dt"] - long["prev_dt"]).dt.days
rest_home = long.rename(columns={"team_id": "home_team_id", "rest_days": "home_rest_days"})
rest_away = long.rename(columns={"team_id": "away_team_id", "rest_days": "away_rest_days"})
matches = matches.merge(rest_home[["match_id", "home_team_id", "home_rest_days"]], on=["match_id", "home_team_id"], how="left")
matches = matches.merge(rest_away[["match_id", "away_team_id", "away_rest_days"]], on=["match_id", "away_team_id"], how="left")

# Host flags
team_name_map = teams_db.set_index("team_id")["team_name"]
matches["home_name"]        = matches["home_team_id"].map(team_name_map)
matches["away_name"]        = matches["away_team_id"].map(team_name_map)
matches["feat_home_is_host"] = (_norm(matches["home_name"]) == _norm(matches["host_country"].fillna(""))).astype(int)
matches["feat_away_is_host"] = (_norm(matches["away_name"]) == _norm(matches["host_country"].fillna(""))).astype(int)

# Team feature joins in home/away form
ft_home = feat_team.add_prefix("home_").rename(columns={"home_tournament_id": "tournament_id", "home_team_id": "home_team_id"})
ft_away = feat_team.add_prefix("away_").rename(columns={"away_tournament_id": "tournament_id", "away_team_id": "away_team_id"})
df_all  = matches.merge(ft_home, on=["tournament_id", "home_team_id"], how="left")
df_all  = df_all.merge(ft_away, on=["tournament_id", "away_team_id"], how="left")
df_all["feat_same_confederation"] = (df_all["home_confederation_id"] == df_all["away_confederation_id"]).astype(int)

# Favorite / underdog framing (same as 06)
home_elo    = df_all["home_elo_rating"].fillna(0)
away_elo    = df_all["away_elo_rating"].fillna(0)
home_is_fav = home_elo >= away_elo
team_cols   = [c for c in feat_team.columns if c not in ("tournament_id", "team_id", "year", "team_name", "team_code", "confederation_id")]

fav_rows = []
for i, (_, row) in enumerate(df_all.iterrows()):
    fav_side = "home" if home_is_fav.iloc[i] else "away"
    und_side = "away" if fav_side == "home" else "home"
    r = {
        "match_id": row["match_id"], "tournament_id": row["tournament_id"], "year": row["year"],
        "group_stage": row.get("group_stage", np.nan), "knockout_stage": row.get("knockout_stage", np.nan),
        "feat_home_is_host": row["feat_home_is_host"], "feat_away_is_host": row["feat_away_is_host"],
        "feat_same_confederation": row["feat_same_confederation"],
        "fav_rest_days": row.get(f"{fav_side}_rest_days", np.nan),
        "und_rest_days": row.get(f"{und_side}_rest_days", np.nan),
        "fav_is_home": int(fav_side == "home"),
        "fav_team_id": row[f"{fav_side}_team_id"], "und_team_id": row[f"{und_side}_team_id"],
    }
    for col in team_cols:
        r[f"fav_{col}"] = row.get(f"{fav_side}_{col}", np.nan)
        r[f"und_{col}"] = row.get(f"{und_side}_{col}", np.nan)
    fav_win = row.get(f"{fav_side}_team_win", np.nan)
    draw    = row.get("draw", np.nan)
    r["y"]  = np.nan if pd.isna(fav_win) else (1 if draw == 1 else (2 if fav_win == 1 else 0))
    fav_rows.append(r)

df_fav = pd.DataFrame(fav_rows).dropna(subset=["y"]).copy()
df_fav["y"] = df_fav["y"].astype(int)

# Player/squad joins
ps_cols = [c for c in feat_ps.columns if c not in ("tournament_id", "team_id", "national_team", "year")]
ps_fav  = feat_ps.rename(columns={c: f"fav_{c}" for c in ps_cols}).rename(columns={"team_id": "fav_team_id"})
ps_und  = feat_ps.rename(columns={c: f"und_{c}" for c in ps_cols}).rename(columns={"team_id": "und_team_id"})
df_fav  = df_fav.merge(ps_fav[["tournament_id", "fav_team_id"] + [f"fav_{c}" for c in ps_cols]], on=["tournament_id", "fav_team_id"], how="left")
df_fav  = df_fav.merge(ps_und[["tournament_id", "und_team_id"] + [f"und_{c}" for c in ps_cols]], on=["tournament_id", "und_team_id"], how="left")

# Derived features
df_fav["elo_gap"]          = df_fav["fav_elo_rating"] - df_fav["und_elo_rating"]
df_fav["hist_win_rate_diff"]  = df_fav["fav_hist_win_rate_shrunk"] - df_fav["und_hist_win_rate_shrunk"]
df_fav["hist_draw_rate_diff"] = df_fav["fav_hist_draw_rate_shrunk"] - df_fav["und_hist_draw_rate_shrunk"]
df_fav["rest_days_diff"]   = df_fav["fav_rest_days"] - df_fav["und_rest_days"]

print("df_fav shape:", df_fav.shape)
print("Years:", sorted(df_fav["year"].dropna().astype(int).unique()))
display(df_fav["y"].value_counts().sort_index().rename(index={0: "fav_loses", 1: "draw", 2: "fav_wins"}).to_frame("count"))

## B. Feature sets (identical to 06)

In [ ]:
TEAM_HIST_COLS = [
    "fav_hist_win_rate_shrunk", "und_hist_win_rate_shrunk",
    "fav_hist_draw_rate_shrunk", "und_hist_draw_rate_shrunk",
    "fav_hist_ko_win_rate_shrunk", "und_hist_ko_win_rate_shrunk",
    "fav_hist_goal_diff_per_match", "und_hist_goal_diff_per_match",
    "fav_hist_frac_ko", "und_hist_frac_ko",
    "fav_hist_n_tournaments", "und_hist_n_tournaments",
    "fav_hist_pso_win_rate_shrunk", "und_hist_pso_win_rate_shrunk",
    "hist_win_rate_diff", "hist_draw_rate_diff", "elo_gap",
]
TEAM_SQUAD_COLS = [
    "fav_squad_age_mean", "und_squad_age_mean",
    "fav_squad_prior_wc_mean", "und_squad_prior_wc_mean",
    "fav_squad_share_any_prior_wc", "und_squad_share_any_prior_wc",
    "fav_squad_jaccard_vs_prev_wc", "und_squad_jaccard_vs_prev_wc",
]
TEAM_COACH_COLS = [
    "fav_manager_local", "und_manager_local",
    "fav_mgr_n_prior_wc", "und_mgr_n_prior_wc",
    "fav_mgr_hist_win_rate_shrunk", "und_mgr_hist_win_rate_shrunk",
    "fav_mgr_consecutive_wc_with_team", "und_mgr_consecutive_wc_with_team",
]
TEAM_RANK_COLS = [
    "fav_elo_rating", "und_elo_rating",
    "fav_fifa_rank", "und_fifa_rank",
    "fav_fifa_points", "und_fifa_points",
]
MATCH_COLS = [
    "fav_is_home", "group_stage", "knockout_stage",
    "feat_home_is_host", "feat_away_is_host", "feat_same_confederation",
    "rest_days_diff",
]
PLAYER_ONLY_COLS = [
    c for c in df_fav.columns
    if c.startswith("fav_pl_") or c.startswith("und_pl_")
    or c.startswith("fav_sq_") or c.startswith("und_sq_")
]

TEAM_ONLY_FEATURES = [
    c for c in (TEAM_HIST_COLS + TEAM_SQUAD_COLS + TEAM_COACH_COLS + TEAM_RANK_COLS + MATCH_COLS)
    if c in df_fav.columns
]
PLAYER_ENRICHED_FEATURES = [
    c for c in list(dict.fromkeys(TEAM_ONLY_FEATURES + PLAYER_ONLY_COLS))
    if c in df_fav.columns
]

print(f"TEAM_ONLY_FEATURES     : {len(TEAM_ONLY_FEATURES)} features")
print(f"PLAYER_ENRICHED_FEATURES: {len(PLAYER_ENRICHED_FEATURES)} features")

# Verify feature counts match 06 locked specs
specs_06 = json.loads((LOCK_DIR / "final_model_specs.json").read_text())
assert len(TEAM_ONLY_FEATURES) == specs_06["baseline"]["n_features"], \
    f"Team-only count mismatch: {len(TEAM_ONLY_FEATURES)} vs {specs_06['baseline']['n_features']}"
assert len(PLAYER_ENRICHED_FEATURES) == specs_06["player_enriched"]["n_features"], \
    f"Player-enriched count mismatch: {len(PLAYER_ENRICHED_FEATURES)} vs {specs_06['player_enriched']['n_features']}"
print("Feature count parity with 06 confirmed.")

## C. Unified XGBoost model configs

All three models use **the same XGBoost hyperparameters** as the `player_enriched` model in `06`. The only differences are (a) training window and (b) feature set — which is exactly the experimental design we want to test.

In [ ]:
SHARED_XGB_PARAMS = dict(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=SEED,
    verbosity=0,
    n_jobs=-1,
)

def make_xgb_pipeline():
    return Pipeline([
        ("imp", SimpleImputer()),
        ("clf", xgb.XGBClassifier(**SHARED_XGB_PARAMS)),
    ])

UNIFIED_CONFIGS = {
    "baseline": {
        "display_name": "Baseline (XGB)",
        "training_window": "2006-2014",
        "train_min_year": 2006,
        "main_signals": "Team-level only",
        "features": TEAM_ONLY_FEATURES,
    },
    "team_history": {
        "display_name": "Team-history (XGB)",
        "training_window": "1998-2014",
        "train_min_year": 1998,
        "main_signals": "Team-level only",
        "features": TEAM_ONLY_FEATURES,
    },
    "player_enriched": {
        "display_name": "Player-enriched (XGB)",
        "training_window": "2006-2014",
        "train_min_year": 2006,
        "main_signals": "Team + player/squad",
        "features": PLAYER_ENRICHED_FEATURES,
    },
}

display(pd.DataFrame([
    {"model_key": k, "display_name": v["display_name"],
     "training_window": v["training_window"], "n_features": len(v["features"])}
    for k, v in UNIFIED_CONFIGS.items()
]))

## D. Helper functions (LOTO CV + holdout evaluation)

In [ ]:
def loto_cv(df_train, feature_cols, pipeline):
    years = sorted(df_train["year"].unique())
    accs, f1s = [], []
    for test_year in years:
        tr = df_train[df_train["year"] != test_year]
        te = df_train[df_train["year"] == test_year]
        m  = clone(pipeline)
        m.fit(tr[feature_cols], tr["y"])
        pred = m.predict(te[feature_cols])
        accs.append(accuracy_score(te["y"], pred))
        f1s.append(f1_score(te["y"], pred, average="macro", zero_division=0))
    return {"cv_acc": float(np.mean(accs)), "cv_acc_std": float(np.std(accs)),
            "cv_f1": float(np.mean(f1s)), "cv_f1_std": float(np.std(f1s)), "cv_years": years}


def fit_and_predict_holdout(df, feature_cols, pipeline, train_min_year):
    train   = df[(df["year"] >= train_min_year) & (~df["year"].isin(HOLDOUT_YEARS))].copy()
    holdout = df[df["year"].isin(HOLDOUT_YEARS)].copy()
    m = clone(pipeline)
    m.fit(train[feature_cols], train["y"])
    pred = m.predict(holdout[feature_cols])
    prob = m.predict_proba(holdout[feature_cols])
    out  = holdout[["match_id", "tournament_id", "year", "y",
                     "group_stage", "knockout_stage",
                     "fav_team_id", "und_team_id"]].copy()
    out["pred"]           = pred
    out["prob_fav_loses"] = prob[:, 0]
    out["prob_draw"]      = prob[:, 1]
    out["prob_fav_wins"]  = prob[:, 2]
    metrics = {
        "holdout_acc":    float(accuracy_score(out["y"], out["pred"])),
        "holdout_f1":     float(f1_score(out["y"], out["pred"], average="macro", zero_division=0)),
    }
    return m, out, metrics


def stage_breakdown(pred_df):
    rows = []
    masks = {
        "group_stage":    pred_df["group_stage"].fillna(0).astype(bool),
        "knockout_stage": pred_df["knockout_stage"].fillna(0).astype(bool),
        "overall":        pd.Series(True, index=pred_df.index),
    }
    for stage, mask in masks.items():
        sub = pred_df[mask]
        rows.append({
            "stage": stage, "n": int(len(sub)),
            "accuracy": float(accuracy_score(sub["y"], sub["pred"])),
            "macro_f1": float(f1_score(sub["y"], sub["pred"], average="macro", zero_division=0)),
        })
    return pd.DataFrame(rows)

## E. Train all three unified-XGB models and evaluate

In [ ]:
results_xgb = {}

for model_key, cfg in UNIFIED_CONFIGS.items():
    pipe = make_xgb_pipeline()
    train_df = df_fav[
        (df_fav["year"] >= cfg["train_min_year"]) & (~df_fav["year"].isin(HOLDOUT_YEARS))
    ].copy()

    print(f"\n--- {cfg['display_name']} | n_train={len(train_df)} ---")
    cv      = loto_cv(train_df, cfg["features"], pipe)
    _, pred_df, metrics = fit_and_predict_holdout(df_fav, cfg["features"], pipe, cfg["train_min_year"])
    stage   = stage_breakdown(pred_df)

    results_xgb[model_key] = {
        "cfg": cfg, "cv": cv, "pred_df": pred_df,
        "metrics": metrics, "stage": stage,
    }
    print(f"  LOTO CV  acc={cv['cv_acc']:.3f} ± {cv['cv_acc_std']:.3f}  "
          f"macro-F1={cv['cv_f1']:.3f} ± {cv['cv_f1_std']:.3f}")
    print(f"  Holdout  acc={metrics['holdout_acc']:.3f}  macro-F1={metrics['holdout_f1']:.3f}")

print("\nAll models trained.")

## F. Side-by-side comparison: 06 (RF/XGB mixed) vs 10 (unified XGB)

Read the 06 locked metrics and place them next to the unified-XGB results so we can judge whether the core conclusions change.

In [ ]:
table1_06 = pd.read_csv(LOCK_DIR / "table1_summary.csv", index_col=0)

compare_rows = []
for key in MODEL_KEYS:
    r06 = table1_06.loc[key]
    r10 = results_xgb[key]
    compare_rows.append({
        "model_key":        key,
        "06_algorithm":     r06["model_family"],
        "06_cv_acc":        r06["cv_accuracy"],
        "06_holdout_acc":   r06["holdout_accuracy"],
        "06_holdout_f1":    r06["holdout_macro_f1"],
        "10_algorithm":     "XGBClassifier",
        "10_cv_acc":        r10["cv"]["cv_acc"],
        "10_holdout_acc":   r10["metrics"]["holdout_acc"],
        "10_holdout_f1":    r10["metrics"]["holdout_f1"],
        "delta_acc":        r10["metrics"]["holdout_acc"]  - r06["holdout_accuracy"],
        "delta_f1":         r10["metrics"]["holdout_f1"]   - r06["holdout_macro_f1"],
    })

compare_df = pd.DataFrame(compare_rows).set_index("model_key")
display(compare_df.style.format({
    c: "{:.3f}" for c in compare_df.columns if c not in ("06_algorithm", "10_algorithm")
}).set_caption("06 (mixed RF/XGB) vs 10 (unified XGB) — holdout metrics"))

## G. Paired bootstrap on unified-XGB predictions

Same procedure as `07`, applied to the unified-XGB holdout predictions.

In [ ]:
# Merge all three prediction columns into one holdout frame
holdout_all = results_xgb["baseline"]["pred_df"][
    ["match_id", "year", "y", "group_stage", "knockout_stage"]
].copy()
for key in MODEL_KEYS:
    p = results_xgb[key]["pred_df"][["match_id", "pred", "prob_fav_loses", "prob_draw", "prob_fav_wins"]]
    p = p.rename(columns={
        "pred":           f"{key}_pred",
        "prob_fav_loses": f"{key}_prob_fav_loses",
        "prob_draw":      f"{key}_prob_draw",
        "prob_fav_wins":  f"{key}_prob_fav_wins",
    })
    holdout_all = holdout_all.merge(p, on="match_id", how="left")

print("Holdout frame shape:", holdout_all.shape)

def _eval(df, key):
    return {
        "accuracy": accuracy_score(df["y"], df[f"{key}_pred"]),
        "macro_f1": f1_score(df["y"], df[f"{key}_pred"], average="macro", zero_division=0),
    }

def bootstrap_pairwise(df, hi="player_enriched", lo="team_history",
                        stage_col=None, b=BOOTSTRAP_B, seed=SEED):
    rng   = np.random.default_rng(seed)
    mask  = df[stage_col].fillna(0).astype(bool) if stage_col else pd.Series(True, index=df.index)
    base  = df[mask]
    idx   = np.arange(len(base))
    n     = len(base)
    hi_acc, hi_f1, lo_acc, lo_f1, d_acc, d_f1 = [], [], [], [], [], []

    for _ in range(b):
        s = base.iloc[rng.choice(idx, size=n, replace=True)]
        ha, hf = _eval(s, hi)["accuracy"], _eval(s, hi)["macro_f1"]
        la, lf = _eval(s, lo)["accuracy"], _eval(s, lo)["macro_f1"]
        hi_acc.append(ha); hi_f1.append(hf)
        lo_acc.append(la); lo_f1.append(lf)
        d_acc.append(ha - la); d_f1.append(hf - lf)

    def ci(v):
        v = np.array(v)
        return {"mean": float(np.mean(v)), "ci_low": float(np.percentile(v, 2.5)),
                "ci_high": float(np.percentile(v, 97.5))}

    pt_hi = _eval(base, hi)
    pt_lo = _eval(base, lo)
    return {
        "n": n,
        f"{hi}_acc":  {**ci(hi_acc), "point": pt_hi["accuracy"]},
        f"{hi}_f1":   {**ci(hi_f1),  "point": pt_hi["macro_f1"]},
        f"{lo}_acc":  {**ci(lo_acc), "point": pt_lo["accuracy"]},
        f"{lo}_f1":   {**ci(lo_f1),  "point": pt_lo["macro_f1"]},
        "diff_acc":   {**ci(d_acc),  "point": pt_hi["accuracy"]  - pt_lo["accuracy"]},
        "diff_f1":    {**ci(d_f1),   "point": pt_hi["macro_f1"]  - pt_lo["macro_f1"]},
    }

bs_overall   = bootstrap_pairwise(holdout_all)
bs_group     = bootstrap_pairwise(holdout_all, stage_col="group_stage")
bs_knockout  = bootstrap_pairwise(holdout_all, stage_col="knockout_stage")

def fmt_row(label, bs, metric):
    pt  = bs[f"diff_{metric}"]["point"]
    lo  = bs[f"diff_{metric}"]["ci_low"]
    hi  = bs[f"diff_{metric}"]["ci_high"]
    sig = "CI excludes 0" if (lo > 0 or hi < 0) else "CI includes 0"
    return {"stage": label, "n": bs["n"],
            f"player_enriched_{metric}": round(bs[f"player_enriched_{metric}"]["point"], 3),
            f"team_history_{metric}":    round(bs[f"team_history_{metric}"]["point"],    3),
            f"diff_{metric} [95% CI]": f"{pt:+.3f} [{lo:+.3f}, {hi:+.3f}]",
            "note": sig}

for metric in ["acc", "f1"]:
    rows = [
        fmt_row("group_stage",    bs_group,    metric),
        fmt_row("knockout_stage", bs_knockout, metric),
        fmt_row("overall",        bs_overall,  metric),
    ]
    print(f"\nPlayer-enriched vs Team-history — {metric} (unified XGB)")
    display(pd.DataFrame(rows))

## H. Verdict: do the core report conclusions hold under unified XGB?

In [ ]:
r_bl = results_xgb["baseline"]
r_th = results_xgb["team_history"]
r_pe = results_xgb["player_enriched"]

conclusions = [
    {
        "claim": "player_enriched macro-F1 ≥ team_history (overall)",
        "holds": r_pe["metrics"]["holdout_f1"] >= r_th["metrics"]["holdout_f1"],
        "value_10": f"{r_pe['metrics']['holdout_f1']:.3f} vs {r_th['metrics']['holdout_f1']:.3f}",
        "value_06": f"{table1_06.loc['player_enriched','holdout_macro_f1']:.3f} vs "
                    f"{table1_06.loc['team_history','holdout_macro_f1']:.3f}",
    },
    {
        "claim": "overall accuracy within 5pp across all three models",
        "holds": (
            max(r_bl["metrics"]["holdout_acc"],
                r_th["metrics"]["holdout_acc"],
                r_pe["metrics"]["holdout_acc"]) -
            min(r_bl["metrics"]["holdout_acc"],
                r_th["metrics"]["holdout_acc"],
                r_pe["metrics"]["holdout_acc"])
        ) <= 0.05,
        "value_10": (f"bl={r_bl['metrics']['holdout_acc']:.3f}  "
                     f"th={r_th['metrics']['holdout_acc']:.3f}  "
                     f"pe={r_pe['metrics']['holdout_acc']:.3f}"),
        "value_06": (f"bl={table1_06.loc['baseline','holdout_accuracy']:.3f}  "
                     f"th={table1_06.loc['team_history','holdout_accuracy']:.3f}  "
                     f"pe={table1_06.loc['player_enriched','holdout_accuracy']:.3f}"),
    },
    {
        "claim": "player_enriched knockout accuracy > team_history knockout accuracy",
        "holds": (
            r_pe["stage"].set_index("stage").loc["knockout_stage", "accuracy"] >
            r_th["stage"].set_index("stage").loc["knockout_stage", "accuracy"]
        ),
        "value_10": (
            f"pe={r_pe['stage'].set_index('stage').loc['knockout_stage','accuracy']:.3f}  "
            f"th={r_th['stage'].set_index('stage').loc['knockout_stage','accuracy']:.3f}"
        ),
        "value_06": "pe=0.781  th=0.719",
    },
    {
        "claim": "player_enriched group-stage accuracy ≈ team_history (within 3pp)",
        "holds": abs(
            r_pe["stage"].set_index("stage").loc["group_stage", "accuracy"] -
            r_th["stage"].set_index("stage").loc["group_stage", "accuracy"]
        ) <= 0.03,
        "value_10": (
            f"pe={r_pe['stage'].set_index('stage').loc['group_stage','accuracy']:.3f}  "
            f"th={r_th['stage'].set_index('stage').loc['group_stage','accuracy']:.3f}"
        ),
        "value_06": "pe=0.615  th=0.625",
    },
]

verdict_df = pd.DataFrame(conclusions)
verdict_df["status"] = verdict_df["holds"].map({True: "HOLDS", False: "FAILS"})
display(verdict_df[["claim", "status", "value_06", "value_10"]])

all_hold = verdict_df["holds"].all()
print("\n" + ("=" * 60))
if all_hold:
    print("CONCLUSION: All core claims hold under unified XGB.")
    print("The original conclusions are attributable to data differences,")
    print("not to the RF vs XGB algorithm switch.")
    print("=> SAFE to update report with unified-algorithm framing.")
else:
    failed = verdict_df[~verdict_df["holds"]]["claim"].tolist()
    print("CONCLUSION: The following claims DO NOT hold under unified XGB:")
    for f in failed:
        print(f"  - {f}")
    print("=> Report narrative needs revision before updating algorithm framing.")
print("=" * 60)

## I. Export sensitivity results

In [ ]:
compare_df.to_csv(OUT_DIR / "model_comparison_06_vs_10.csv")
verdict_df[["claim", "status", "value_06", "value_10"]].to_csv(OUT_DIR / "verdict.csv", index=False)

# Bootstrap summary table
bs_rows = []
for stage_label, bs in [("group_stage", bs_group), ("knockout_stage", bs_knockout), ("overall", bs_overall)]:
    for metric, col in [("accuracy", "acc"), ("macro_f1", "f1")]:
        bs_rows.append({
            "stage": stage_label,
            "metric": metric,
            "n": bs["n"],
            "player_enriched_point": bs[f"player_enriched_{col}"]["point"],
            "team_history_point":    bs[f"team_history_{col}"]["point"],
            "diff_point":            bs[f"diff_{col}"]["point"],
            "diff_ci_low":           bs[f"diff_{col}"]["ci_low"],
            "diff_ci_high":          bs[f"diff_{col}"]["ci_high"],
        })
bs_df = pd.DataFrame(bs_rows)
bs_df.to_csv(OUT_DIR / "bootstrap_pairwise_unified_xgb.csv", index=False)

print("Saved to:", OUT_DIR)
for f in sorted(OUT_DIR.iterdir()):
    print(" ", f.name)

## J. Unified Random Forest sensitivity check (Option A)

XGB without explicit class balancing behaves differently from RF with `class_weight='balanced'`, which makes the Section F/G comparison confounded by two things at once (algorithm + class-weighting scheme). This section runs a second sensitivity check where **all three models use RF with `class_weight='balanced'`** — the algorithm used by baseline and team-history in `06`. Player-enriched with RF is the new test.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

SHARED_RF_PARAMS = dict(
    n_estimators=300,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)

def make_rf_pipeline():
    return Pipeline([
        ("imp", SimpleImputer()),
        ("clf", RandomForestClassifier(**SHARED_RF_PARAMS)),
    ])

RF_CONFIGS = {
    "baseline": {
        "display_name": "Baseline (RF)",
        "training_window": "2006-2014",
        "train_min_year": 2006,
        "features": TEAM_ONLY_FEATURES,
    },
    "team_history": {
        "display_name": "Team-history (RF)",
        "training_window": "1998-2014",
        "train_min_year": 1998,
        "features": TEAM_ONLY_FEATURES,
    },
    "player_enriched": {
        "display_name": "Player-enriched (RF)",   # <-- new: RF instead of XGB
        "training_window": "2006-2014",
        "train_min_year": 2006,
        "features": PLAYER_ENRICHED_FEATURES,
    },
}

results_rf = {}
for model_key, cfg in RF_CONFIGS.items():
    pipe     = make_rf_pipeline()
    train_df = df_fav[
        (df_fav["year"] >= cfg["train_min_year"]) & (~df_fav["year"].isin(HOLDOUT_YEARS))
    ].copy()
    print(f"\n--- {cfg['display_name']} | n_train={len(train_df)} ---")
    cv      = loto_cv(train_df, cfg["features"], pipe)
    _, pred_df, metrics = fit_and_predict_holdout(
        df_fav, cfg["features"], pipe, cfg["train_min_year"]
    )
    stage = stage_breakdown(pred_df)
    results_rf[model_key] = {
        "cfg": cfg, "cv": cv, "pred_df": pred_df, "metrics": metrics, "stage": stage,
    }
    print(f"  LOTO CV  acc={cv['cv_acc']:.3f} ± {cv['cv_acc_std']:.3f}  "
          f"macro-F1={cv['cv_f1']:.3f} ± {cv['cv_f1_std']:.3f}")
    print(f"  Holdout  acc={metrics['holdout_acc']:.3f}  macro-F1={metrics['holdout_f1']:.3f}")

print("\nAll RF models trained.")

## K. Three-way comparison: 06 original vs unified XGB vs unified RF

In [ ]:
three_way_rows = []
for key in MODEL_KEYS:
    r06 = table1_06.loc[key]
    r10 = results_xgb[key]
    rrf = results_rf[key]
    three_way_rows.append({
        "model_key":       key,
        # 06 original
        "06_algo":         r06["model_family"][:3],   # RF / XGB
        "06_acc":          round(r06["holdout_accuracy"],   3),
        "06_f1":           round(r06["holdout_macro_f1"],   3),
        # unified XGB (Section E–G)
        "xgb_acc":         round(r10["metrics"]["holdout_acc"], 3),
        "xgb_f1":          round(r10["metrics"]["holdout_f1"],  3),
        # unified RF (Section J)
        "rf_acc":          round(rrf["metrics"]["holdout_acc"], 3),
        "rf_f1":           round(rrf["metrics"]["holdout_f1"],  3),
    })

three_way = pd.DataFrame(three_way_rows).set_index("model_key")
display(three_way.style.set_caption(
    "Holdout metrics: 06 original | unified XGB | unified RF\n"
    "(bold = best per metric column)"
))

## L. Paired bootstrap: unified RF (player_enriched vs team_history)

In [ ]:
# Merge RF predictions into one holdout frame
holdout_rf = results_rf["baseline"]["pred_df"][
    ["match_id", "year", "y", "group_stage", "knockout_stage"]
].copy()
for key in MODEL_KEYS:
    p = results_rf[key]["pred_df"][
        ["match_id", "pred", "prob_fav_loses", "prob_draw", "prob_fav_wins"]
    ].rename(columns={
        "pred":           f"{key}_pred",
        "prob_fav_loses": f"{key}_prob_fav_loses",
        "prob_draw":      f"{key}_prob_draw",
        "prob_fav_wins":  f"{key}_prob_fav_wins",
    })
    holdout_rf = holdout_rf.merge(p, on="match_id", how="left")

bs_rf_overall  = bootstrap_pairwise(holdout_rf)
bs_rf_group    = bootstrap_pairwise(holdout_rf, stage_col="group_stage")
bs_rf_knockout = bootstrap_pairwise(holdout_rf, stage_col="knockout_stage")

for metric, col in [("accuracy", "acc"), ("macro_f1", "f1")]:
    rows = [
        fmt_row("group_stage",    bs_rf_group,    col),
        fmt_row("knockout_stage", bs_rf_knockout, col),
        fmt_row("overall",        bs_rf_overall,  col),
    ]
    print(f"\nPlayer-enriched vs Team-history — {metric} (unified RF)")
    display(pd.DataFrame(rows))

# Stage-wise accuracy table for quick inspection
print("\nStage-wise accuracy — unified RF")
stage_rf_rows = []
for key in ["team_history", "player_enriched"]:
    for _, row in results_rf[key]["stage"].iterrows():
        stage_rf_rows.append({
            "model": RF_CONFIGS[key]["display_name"],
            "stage": row["stage"], "n": row["n"],
            "accuracy": round(row["accuracy"], 3),
            "macro_f1": round(row["macro_f1"],  3),
        })
display(pd.DataFrame(stage_rf_rows))

## M. Verdict — unified RF

In [ ]:
r_bl_rf = results_rf["baseline"]
r_th_rf = results_rf["team_history"]
r_pe_rf = results_rf["player_enriched"]

pe_ko  = r_pe_rf["stage"].set_index("stage").loc["knockout_stage", "accuracy"]
th_ko  = r_th_rf["stage"].set_index("stage").loc["knockout_stage", "accuracy"]
pe_gs  = r_pe_rf["stage"].set_index("stage").loc["group_stage",    "accuracy"]
th_gs  = r_th_rf["stage"].set_index("stage").loc["group_stage",    "accuracy"]

conclusions_rf = [
    {
        "claim": "player_enriched macro-F1 ≥ team_history (overall)",
        "holds": r_pe_rf["metrics"]["holdout_f1"] >= r_th_rf["metrics"]["holdout_f1"],
        "value_06": f"pe={table1_06.loc['player_enriched','holdout_macro_f1']:.3f}  "
                    f"th={table1_06.loc['team_history','holdout_macro_f1']:.3f}",
        "value_rf": f"pe={r_pe_rf['metrics']['holdout_f1']:.3f}  "
                    f"th={r_th_rf['metrics']['holdout_f1']:.3f}",
    },
    {
        "claim": "overall accuracy within 5pp across all three models",
        "holds": (
            max(r_bl_rf["metrics"]["holdout_acc"],
                r_th_rf["metrics"]["holdout_acc"],
                r_pe_rf["metrics"]["holdout_acc"]) -
            min(r_bl_rf["metrics"]["holdout_acc"],
                r_th_rf["metrics"]["holdout_acc"],
                r_pe_rf["metrics"]["holdout_acc"])
        ) <= 0.05,
        "value_06": "bl=0.656  th=0.648  pe=0.656",
        "value_rf": (f"bl={r_bl_rf['metrics']['holdout_acc']:.3f}  "
                     f"th={r_th_rf['metrics']['holdout_acc']:.3f}  "
                     f"pe={r_pe_rf['metrics']['holdout_acc']:.3f}"),
    },
    {
        "claim": "player_enriched knockout accuracy > team_history knockout accuracy",
        "holds": pe_ko > th_ko,
        "value_06": "pe=0.781  th=0.719",
        "value_rf": f"pe={pe_ko:.3f}  th={th_ko:.3f}",
    },
    {
        "claim": "player_enriched group-stage accuracy ≈ team_history (within 3pp)",
        "holds": abs(pe_gs - th_gs) <= 0.03,
        "value_06": "pe=0.615  th=0.625",
        "value_rf": f"pe={pe_gs:.3f}  th={th_gs:.3f}",
    },
]

verdict_rf = pd.DataFrame(conclusions_rf)
verdict_rf["status"] = verdict_rf["holds"].map({True: "HOLDS", False: "FAILS"})
display(verdict_rf[["claim", "status", "value_06", "value_rf"]])

all_hold_rf = verdict_rf["holds"].all()
print("\n" + ("=" * 60))
if all_hold_rf:
    print("CONCLUSION (unified RF): All core claims HOLD.")
    print("Player-feature advantage is attributable to data, not algorithm.")
    print("=> SAFE to replace 06 player_enriched (XGB) with RF version,")
    print("   and update the report with a clean algorithmic comparison.")
else:
    failed_rf = verdict_rf[~verdict_rf["holds"]]["claim"].tolist()
    print("CONCLUSION (unified RF): The following claims still FAIL:")
    for f in failed_rf:
        print(f"  - {f}")
    print("=> Player-feature advantage does not survive algorithm unification.")
    print("   Consider revising the report narrative accordingly.")
print("=" * 60)

## N. Export unified-RF results

In [ ]:
verdict_rf[["claim", "status", "value_06", "value_rf"]].to_csv(
    OUT_DIR / "verdict_unified_rf.csv", index=False
)
three_way.to_csv(OUT_DIR / "three_way_comparison.csv")

# Bootstrap summary for unified RF
bs_rf_rows = []
for stage_label, bs in [
    ("group_stage",    bs_rf_group),
    ("knockout_stage", bs_rf_knockout),
    ("overall",        bs_rf_overall),
]:
    for metric, col in [("accuracy", "acc"), ("macro_f1", "f1")]:
        bs_rf_rows.append({
            "stage": stage_label, "metric": metric, "n": bs["n"],
            "player_enriched_point": bs[f"player_enriched_{col}"]["point"],
            "team_history_point":    bs[f"team_history_{col}"]["point"],
            "diff_point":            bs[f"diff_{col}"]["point"],
            "diff_ci_low":           bs[f"diff_{col}"]["ci_low"],
            "diff_ci_high":          bs[f"diff_{col}"]["ci_high"],
        })
pd.DataFrame(bs_rf_rows).to_csv(OUT_DIR / "bootstrap_pairwise_unified_rf.csv", index=False)

print("Saved to:", OUT_DIR)
for f in sorted(OUT_DIR.iterdir()):
    print(" ", f.name)